# 🆘 Daily Challenge — Emergency Medical Dispatch AI Agent

**Goal:** Design an AI agent that triages 911 emergency calls, gathers critical medical information, and dispatches the correct medical services.

**You Will Learn:**
- How AI agents perceive → reason → act  
- How agent architectures differ (Reactive, Deliberative, Hybrid)  
- How to choose & integrate tools (APIs, dispatch systems, GIS)  
- How to manage state (symptoms, caller info, action history)  
- How to evaluate agent performance

**You Will Create:**
- A fully defined EMR Dispatch Agent  
- Architecture + tools + state schema  
- Decision-making pipeline  
- Comparison between two agent types

# 1. Understand the Scenario

We are designing an AI assistant that supports emergency medical dispatchers by:

- Analyzing caller symptoms  
- Determining urgency  
- Recommending or initiating medical response  
- Maintaining information throughout the interaction  
- Connecting to dispatch tools and medical triage systems

This agent must operate in **high-risk, time-critical environments** where accuracy, safety, and reasoning quality matter.

## 1. Understand the Scenario — Answer

The task is to design an AI agent that supports a 911 emergency medical dispatcher.

The agent must:
- Collect and interpret caller symptoms (voice or text)
- Determine medical urgency
- Provide instructions or dispatch medical services
- Maintain an internal memory throughout the call
- Integrate with real emergency response systems (APIs, scheduling tools)

The core objective is to create a reliable, safe, and fast AI that augments human dispatchers during time-critical emergencies.

# 2. Define the Agent’s Environment (Perception)

The agent receives the following *inputs*:

- **Caller Symptom Description**  
  - Voice transcript  
  - Text message  
  - NLP-extracted key phrases

- **Caller Location**  
  - GPS coordinates  
  - Address  
  - Cell tower approximation

- **Caller Identity & History (if available)**  
  - Phone number  
  - Medical records  
  - Prior emergency calls  

These inputs form the agent’s “environment” — what it can observe and reason about.

## 2. Define the Agent’s Environment — Answer

The agent perceives the following inputs:

### 📥 Caller Symptom Information
- Transcribed voice call
- Typed text (SMS/chat)
- Extracted medical keywords and severity indicators

### 📍 Location Data
- GPS coordinates (if enabled)
- Address or intersection
- Cell-tower or router triangulation fallback

### 🧍 Caller Identity & History (if available)
- Phone number or user ID
- Past emergency call records
- Known medical conditions (e.g., allergies, chronic illness)

These inputs form the dynamic environment from which the agent draws context to determine urgency and recommended actions.

# 3. Select and Describe Tools

The agent must interact with external systems to make safe, accurate decisions.

### 🛠 Required Tools

#### 1. **Symptom Checker API (e.g., Infermedica)**
- **Input:** extracted symptoms, age, gender  
- **Output:** probability of conditions, severity, recommended action

#### 2. **Ambulance Scheduling / Dispatch System**
- **Input:** urgency level, caller location, resource type needed  
- **Output:** confirmation of ambulance dispatch, ETA

#### 3. **Medical Triage LLM (fine-tuned model)**
- **Input:** caller transcript, structured symptom data  
- **Output:** severity score (e.g., 0–1), triage category

Each tool should be described by:
- What data it consumes  
- What it returns  
- When the agent should invoke it

## 3. Select and Describe Tools — Answer

### 1. Symptom Checker API (Example: Infermedica)
- **Consumes:** symptoms, age, sex, onset time, context notes  
- **Returns:** likely conditions, red-flag indicators, severity classification  
- **Role:** provides medical grounding so the agent doesn’t improvise diagnoses.

### 2. Ambulance Dispatch / Scheduling System
- **Consumes:** urgency level, caller location, requested resource (ALS/BLS), timestamp  
- **Returns:** confirmation of dispatch, ETA, crew ID  
- **Role:** handles real-world deployment of medical teams.

### 3. Medical Triage Model (Fine-Tuned LLM)
- **Consumes:** parsed transcript, structured symptom data  
- **Returns:** severity score (0–1), suggested urgency tier  
- **Role:** mediates between raw input and actionable medical interpretation.

### 4. GIS / Mapping Service (Optional)
- **Consumes:** caller location  
- **Returns:** nearest hospitals, travel times, ambulance routes  
- **Role:** improves routing and recommendations.

Each tool enhances accuracy, reduces dispatcher workload, and supports safe decision-making.

# 4. Outline State Management

The agent must maintain a structured memory throughout the call.

### State Must Include:

- **Caller Info**
  - Name, phone number, location
- **Symptoms**
  - Reported symptoms, severity keywords, changes over time
- **Actions Taken**
  - Advice given  
  - API calls made  
  - Dispatch actions  
- **Decision Logs**
  - Severity scores  
  - Threshold comparisons  
  - Final urgency classification

### Example JSON State Schema

```json
{
  "caller": {
    "name": null,
    "phone": null,
    "location": null
  },
  "symptoms": {
    "reported": [],
    "severity_score": null
  },
  "actions": {
    "advice_given": [],
    "dispatch_status": null
  },
  "decision_log": []
}

## 4. Outline State Management — Answer

The agent must maintain a persistent state throughout the interaction so it does not lose critical information.

### 🧠 Required State Components

#### Caller Information
- name (if given)
- phone number
- real-time location

#### Medical Information
- list of reported symptoms
- severity indicators (e.g., “severe chest pain” flagged as critical)
- triage model score

#### Agent Actions
- instructions already given
- escalation actions
- whether an ambulance was dispatched
- tool calls performed

#### Decision History
- timestamped reasoning steps
- thresholds used to classify urgency
- responses provided to the caller

### Example State Schema (JSON)
```json
{
  "caller": {
    "phone": null,
    "location": null
  },
  "symptoms": [],
  "severity_score": null,
  "actions_taken": [],
  "dispatch_status": null,
  "decision_log": []
}

# 5. Design the Decision-Making Process

The agent uses a structured reasoning loop:

---

### 🔍 Step 1 — Parse Symptoms  
Extract:  
- keywords  
- severity indicators  
- time factors  
- red-flag symptoms

---

### 🧠 Step 2 — Query the Triage Model  
The model returns:  
- Severity score (0–1)  
- Condition likelihoods  
- Recommended urgency level

---

### ⚖️ Step 3 — Compare Score Against Thresholds

| Urgency Level | Score Range | Action |
|---------------|-------------|--------|
| **High**      | ≥ 0.75      | Dispatch ambulance immediately |
| **Medium**    | 0.40–0.74   | Direct caller to urgent care |
| **Low**       | < 0.40      | Provide self-care instructions |

---

### 🚑 Step 4 — Agent Actions

- **High →** trigger dispatch API  
- **Medium →** recommend nearest urgent-care facility  
- **Low →** provide validated self-care instructions  

All steps are logged into the agent’s memory/state.

## 5. Design the Decision-Making Process — Answer

The agent’s reasoning loop follows a structured triage pipeline.

---

## 🩺 Step 1: Parse Symptoms
- Use NLP to extract symptoms, duration, context, and severity keywords.
- Flag life-threatening terms such as “not breathing,” “unconscious,” or “severe chest pain.”

---

## 🤖 Step 2: Query the Triage Model
- Input: structured symptom data + transcript excerpt
- Output: numeric severity score (0–1)
- Example: a score of **0.82** indicates high urgency.

---

## ⚖️ Step 3: Apply Threshold Rules

| Urgency Level | Score Range | Action |
|---------------|-------------|--------|
| **High**      | ≥ 0.75      | Immediate ambulance dispatch |
| **Medium**    | 0.40–0.74   | Recommend urgent-care visit |
| **Low**       | < 0.40      | Provide self-care guidance |

---

## 🚑 Step 4: Execute Action
- **High:** trigger dispatch API + stay on the line with caller  
- **Medium:** give clear directions to nearest urgent care + safety checks  
- **Low:** provide validated self-care steps + escalation triggers (“call back if…”)

---

## 📘 Step 5: Log All Decisions
All reasoning, thresholds, and actions are appended to state for auditor review and safety.


# 6. Classify the Agent (Architecture Choice)

**Chosen architecture:** Hybrid Agent

### Justification
A hybrid agent is ideal because:
- It maintains **state** across the call (deliberative behavior).  
- It reacts immediately to symptoms like “not breathing” (reactive behavior).  
- It balances **speed** and **planning**, crucial in emergency medicine.  
- It can integrate multiple tools and adjust decisions as new data arrives.

## 6. Classify Your Agent — Answer

### Chosen Architecture: **Hybrid Agent**

### Why Hybrid?
A hybrid agent combines:
- **Reactive elements** for immediate life-saving responses  
  (e.g., detecting “not breathing” triggers instant escalation)
- **Deliberative planning** for reasoning over symptoms, history, and severity scores  
- **Memory** to track state throughout the call  
- **Tool planning** to decide which external systems to invoke and when

This balance supports both **speed and safety**, which is essential in emergency medical scenarios.

# 7. Compare to a Second Agent Type (Reactive)

### Key Differences

#### 🔁 Reactive Agent
- No long-term planning  
- Acts immediately on current input  
- No memory or state management  
- Fast but risks missing contextual clues

#### 🧠 Hybrid Agent (Your Choice)
- Stores caller data + reasoning steps  
- Can plan multi-step sequences  
- Uses memory + tools more intelligently

### Trade-offs

- **Speed:** Reactive is faster  
- **Reliability:** Hybrid is more accurate  
- **Intelligence:** Hybrid supports better reasoning and tool use

## 7. Compare to a Second Agent Type — Answer

### Comparison: Hybrid Agent vs Reactive Agent

---

## 🟩 Hybrid Agent (Chosen)
- Uses memory + tool reasoning
- Follows a decision pipeline
- Integrates multiple APIs
- More accurate in complex or evolving symptoms
- Slightly slower due to reasoning steps

---

## 🟦 Reactive Agent
- Acts solely based on current input
- No memory or planning
- Fast but shallow:  
  e.g., may not connect multiple mild symptoms into a serious composite condition

---

## ⚖️ Trade-Off Summary

- **Speed:** Reactive is faster  
- **Reliability:** Hybrid is safer and more consistent  
- **Intelligence:** Hybrid interprets context and history, making better decisions  
- **Complexity:** Hybrid requires more engineering and validation

# 8. Reflection Questions

### ❓ What fails if the agent does not maintain state?
Without state, the agent forgets:
- previous symptoms  
- caller identity  
- decisions made  
- whether an ambulance has already been dispatched  

This leads to dangerous inconsistencies, repeated questions, or contradictory advice.

---

### ❓ Why are external tools essential in emergency dispatch?
Dispatch requires:
- Verified medical symptom analysis  
- Real-time scheduling of ambulances  
- Accurate location mapping  

External systems provide **accuracy, real-time data, and high reliability**, which an isolated agent cannot replicate.

These integrations make the agent safe and clinically useful.

## 8. Reflection Answers

## ❓ What fails if your agent does not maintain state?

Without state:
- The agent may repeat questions or forget symptoms.
- It cannot track whether an ambulance was already dispatched.
- Severity assessments become inconsistent.
- Critical decision logs vanish, hurting reliability and legal compliance.
- The agent cannot recognize symptom escalation (e.g., pain worsening).

State is essential for safety, continuity, and accountability.

---

## ❓ Why are external tools essential in an EMR dispatch scenario?

Emergency medicine is high-stakes.  
External tools provide:

- **Medical expertise** (symptom checkers, triage models)  
- **Operational capability** (dispatch systems, ETA calculation)  
- **Geographic awareness** (GIS routing, nearest hospitals)  

These systems ensure the agent is grounded in validated data and can take real-world actions.  
An isolated LLM cannot safely perform medical triage without these integrations.